In [0]:
from pyspark.sql.functions import (
    col,
    collect_set,
    datediff,
    when,
    max as _max,
    sum as _sum,
)

from olist_lakehouse.transformations import parse_timestamp

In [0]:
in_catalog = dbutils.widgets.get("in_catalog")
in_schema = dbutils.widgets.get("in_schema")
out_catalog = dbutils.widgets.get("out_catalog")
out_schema = dbutils.widgets.get("out_schema")

In [0]:
orders = spark.read.table(f"{in_catalog}.{in_schema}.tbl_olist_orders_dataset")
customers = spark.read.table(f"{in_catalog}.{in_schema}.tbl_olist_customers_dataset")
payments = spark.read.table(f"{in_catalog}.{in_schema}.tbl_olist_order_payments_dataset")

## Payments aggregation and data type treatment

In [0]:
payments_treated_types = (
    payments
    .withColumn("payment_value", col("payment_value").cast("double"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
)

In [0]:
payments_agg = (
    payments_treated_types.groupBy("order_id")
    .agg(
        _sum(col("payment_value")).alias("total_payment_value"),
        _max(col("payment_installments")).alias("max_installments"),
        collect_set(col("payment_type")).alias("payment_types_used"),
    )
    .withColumn("payment_types_used", col("payment_types_used").cast("string"))
)

## JOIN and transform silver_orders table

In [0]:
orders_join = orders.join(customers, on="customer_id", how="left").join(payments_agg, on="order_id", how="left")

In [0]:
orders_transformed = orders_join.select(
    "order_id",
    "customer_unique_id",
    "customer_state",
    "order_status",
    parse_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp"),
    "total_payment_value",
    "max_installments",
    "payment_types_used",
    when(col("order_delivered_customer_date").isNull(), None)
    .otherwise(
        datediff(
            parse_timestamp("order_delivered_customer_date"),
            parse_timestamp("order_estimated_delivery_date"),
        )
    )
    .alias("delivery_delay_days"),
).withColumn("is_late", when(col("delivery_delay_days") > 0, True).otherwise(False))

In [0]:
(
    orders_transformed
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{out_catalog}.{out_schema}.orders")
)